# NEERKATHIR AI: train the tomato model and export it for ExecuTorch
Runs on free Google Colab. Steps: 1 GPU, 2 install, 3 data, 4 train, 5 calibrate, 6 export, 7 download.
**Before you start:** Runtime > Change runtime type > **T4 GPU**.
**Honest warning:** PlantVillage photos are lab-style, and several photos come from the same leaf. A random split puts near-copies in train and test, so the accuracy printed here is **too optimistic**. Do not quote it as field accuracy. Test on your own real photos (step 5b) and say so in the report.

In [ ]:
!nvidia-smi -L

## 2. Install (then Runtime > Restart session, and continue from step 3)

In [ ]:
!pip -q install executorch scikit-learn kaggle
import torch, torchvision; print(torch.__version__, torchvision.__version__)

## 3. Get the data
Kaggle: Settings > API > **Create New Token** downloads `kaggle.json`. Upload it below.
If this dataset name has changed, search Kaggle for "PlantVillage dataset" and replace the name. You need the folder with class folders like `Tomato___Early_blight`.

In [ ]:
import os, glob, shutil, random
from google.colab import files
print("Choose kaggle.json")
files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json'); os.chmod('/root/.kaggle/kaggle.json', 0o600)
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content/pv --unzip
roots = [p for p in glob.glob('/content/pv/**/color', recursive=True)]
root = roots[0]; print('using', root); print(sorted(os.listdir(root))[:40])

## 3b. Build the 6 classes the app expects
`healthy, early_blight, late_blight, septoria, leaf_curl, other_unknown`. The **other_unknown** class holds other tomato diseases and other crops, so the model learns to say "not one of mine". Class names must stay exactly like this (the app maps them).

In [ ]:
CLASSES = ['early_blight','healthy','late_blight','leaf_curl','other_unknown','septoria']   # sorted = ImageFolder order
MAP = {'Tomato___healthy':'healthy','Tomato___Early_blight':'early_blight','Tomato___Late_blight':'late_blight',
       'Tomato___Septoria_leaf_spot':'septoria','Tomato___Tomato_Yellow_Leaf_Curl_Virus':'leaf_curl'}
CAP = 1000; random.seed(0)
buckets = {c: [] for c in CLASSES}
for d in sorted(os.listdir(root)):
    buckets[MAP.get(d, 'other_unknown')] += glob.glob(f'{root}/{d}/*')
shutil.rmtree('data', ignore_errors=True)
for k, v in buckets.items():
    random.shuffle(v); v = v[:CAP]; n = len(v)
    parts = {'train': v[:int(.7*n)], 'val': v[int(.7*n):int(.85*n)], 'test': v[int(.85*n):]}
    for sp, fs in parts.items():
        os.makedirs(f'data/{sp}/{k}', exist_ok=True)
        for f in fs: shutil.copy(f, f'data/{sp}/{k}/')
    print(k, {sp: len(fs) for sp, fs in parts.items()})

## 4. Train MobileNetV3-small (about 10 to 15 minutes on a T4)

In [ ]:
import torch, torch.nn as nn
from torchvision import datasets, transforms, models
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
norm = transforms.Normalize([.485,.456,.406],[.229,.224,.225])
tr_t = transforms.Compose([transforms.RandomResizedCrop(224,(0.6,1)), transforms.RandomHorizontalFlip(), transforms.ColorJitter(.4,.4,.4,.05), transforms.ToTensor(), norm])
ev_t = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(), norm])
ds = {s: datasets.ImageFolder(f'data/{s}', tr_t if s=='train' else ev_t) for s in ['train','val','test']}
assert ds['train'].classes == CLASSES, ds['train'].classes
dl = {s: torch.utils.data.DataLoader(ds[s], batch_size=64, shuffle=(s=='train'), num_workers=2) for s in ds}
model = models.mobilenet_v3_small(weights='DEFAULT'); model.classifier[3] = nn.Linear(model.classifier[3].in_features, len(CLASSES)); model.to(dev)
EPOCHS = 8; opt = torch.optim.AdamW(model.parameters(), 1e-3); sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS); lf = nn.CrossEntropyLoss(label_smoothing=0.05)
def logits_of(loader):
    model.eval(); L, Y = [], []
    with torch.no_grad():
        for x, y in loader: L.append(model(x.to(dev)).cpu()); Y.append(y)
    return torch.cat(L), torch.cat(Y)
best = 0
for e in range(EPOCHS):
    model.train()
    for x, y in dl['train']:
        opt.zero_grad(); lf(model(x.to(dev)), y.to(dev)).backward(); opt.step()
    sch.step(); lg, y = logits_of(dl['val']); acc = (lg.argmax(1)==y).float().mean().item(); print(f'epoch {e+1}/{EPOCHS} val acc {acc:.3f}')
    if acc > best: best = acc; torch.save(model.state_dict(), 'best.pt')
model.load_state_dict(torch.load('best.pt')); print('best val acc', best)

## 5. Test, calibrate and choose the "uncertain" threshold
Temperature scaling makes the confidence more honest. The threshold decides when the app says "uncertain". Use the printed numbers in your report.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
vl, vy = logits_of(dl['val']); tl, ty = logits_of(dl['test'])
logT = torch.zeros(1, requires_grad=True); o = torch.optim.LBFGS([logT], lr=0.05, max_iter=100)
def closure():
    o.zero_grad(); l = nn.functional.cross_entropy(vl/logT.exp(), vy); l.backward(); return l
o.step(closure); T = float(logT.exp()); print('temperature T =', round(T, 3))
vp = torch.softmax(vl/T, 1); conf, pred = vp.max(1); best_t = 0.9
for t in [0.5,0.6,0.7,0.8,0.85,0.9,0.95]:
    m = conf >= t; cov = m.float().mean().item(); acc = (pred[m]==vy[m]).float().mean().item() if m.any() else 0
    print(f'threshold {t}: coverage {cov:.2f}, accuracy on accepted {acc:.3f}')
    if acc >= 0.95 and best_t == 0.9 and t <= 0.9: best_t = t
THR = best_t; print('chosen threshold', THR)
tp = torch.softmax(tl/T, 1); tc, tpred = tp.max(1)
print(classification_report(ty, tpred, target_names=CLASSES, digits=3)); print(confusion_matrix(ty, tpred))
acc_m = tc >= THR; print('TEST (optimistic, see warning): accepted', acc_m.float().mean().item(), 'accuracy on accepted', (tpred[acc_m]==ty[acc_m]).float().mean().item())

### 5b. Your own real photos (do this, it is the honest number)
Make folders like `real_test/early_blight/*.jpg`, `real_test/healthy/*.jpg` (use the same 6 class names; put non-tomato or unclear photos in `other_unknown`). Zip it, upload with the left sidebar Files icon, unzip, then run this cell.

In [ ]:
import os
if os.path.isdir('real_test'):
    rds = datasets.ImageFolder('real_test', ev_t); rl, ry = logits_of(torch.utils.data.DataLoader(rds, batch_size=64))
    names = [rds.classes[i] for i in ry.tolist()]; ry2 = torch.tensor([CLASSES.index(n) for n in names])
    rp = torch.softmax(rl/T, 1); rc, rpred = rp.max(1)
    print('REAL PHOTOS accuracy', (rpred==ry2).float().mean().item(), 'n =', len(ry2))
    print(classification_report(ry2, rpred, labels=list(range(len(CLASSES))), target_names=CLASSES, digits=3, zero_division=0))
else: print('No real_test folder yet. Add your photos before you quote any accuracy.')

## 6. Export for the Arm phone (ExecuTorch + XNNPACK)

In [ ]:
import json
class Calibrated(nn.Module):
    def __init__(s, m, T): super().__init__(); s.m = m; s.T = T
    def forward(s, x): return s.m(x) / s.T
wrapped = Calibrated(model.cpu().eval(), T).eval(); ex = (torch.randn(1,3,224,224),)
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
prog = to_edge_transform_and_lower(torch.export.export(wrapped, ex), partitioner=[XnnpackPartitioner()]).to_executorch()
open('tomato_mobilenet.pte', 'wb').write(prog.buffer)
open('classes.txt', 'w').write('\n'.join(CLASSES)); json.dump({'temperature': T, 'threshold': THR, 'classes': CLASSES}, open('calibration.json', 'w'))
print('saved', os.path.getsize('tomato_mobilenet.pte')/1e6, 'MB')
try:
    from executorch.runtime import Runtime
    from pathlib import Path
    m = Runtime.get().load_program(Path('tomato_mobilenet.pte')).load_method('forward')
    out = m.execute([ex[0]])[0]; print('ExecuTorch output matches PyTorch:', torch.allclose(out, wrapped(ex[0]), atol=1e-3))
except Exception as e: print('Runtime check skipped:', e)

## 7. Download and put on the phone
Download the three files, then copy them to the Vivo Y22.

In [ ]:
from google.colab import files
for f in ['tomato_mobilenet.pte', 'classes.txt', 'calibration.json']: files.download(f)

### Copy to the phone
On the phone: Settings > About > tap *Build number* 7 times > Developer options > USB debugging ON. Install the app once and open it (this creates its folder). On a laptop with adb:
```
adb push tomato_mobilenet.pte /sdcard/Android/data/com.neerkathir.ai/files/
adb push classes.txt /sdcard/Android/data/com.neerkathir.ai/files/
```
(Android 12 file managers cannot open `Android/data`, so use adb.) Then in the app: airplane mode ON, scan, About > Run benchmark.
Put `threshold` from `calibration.json` in `MainActivity.kt` (`threshold()`, now 0.70). The temperature is already baked into the model.